# EvalHub Client SDK Usage Examples

This notebook demonstrates how to use the EvalHub client SDK for interacting with the EvalHub evaluation service.

The SDK provides separate client classes for async and sync operations:
- `AsyncEvalHubClient` - For asynchronous operations (recommended for I/O-bound workloads)
- `SyncEvalHubClient` - For synchronous operations

Both use a **nested resource structure**:
- `client.providers` - Provider operations
- `client.benchmarks` - Benchmark operations
- `client.collections` - Collection operations
- `client.jobs` - Evaluation job operations

## Multi-Tenancy

EvalHub uses the `X-Tenant` header to identify the Kubernetes namespace that scopes all
resources. The SDK lets you set a **default tenant** at client construction time and
**override it per-call** when needed:

```python
# Default tenant for every request
client = SyncEvalHubClient(tenant="test", ...)

# Override on a single call
client.jobs.list(tenant="other-ns")
```

## Setup

### Prerequisites

You need:
1. An OpenShift cluster with EvalHub deployed
2. `oc` CLI logged in to the cluster
3. Access to the **`test`** namespace (used as the tenant throughout this notebook)

### 1. Get your authentication token and the EvalHub route

Run the following in a terminal **before** starting Jupyter:

```bash
export EVALHUB_TOKEN=$(oc whoami -t)
echo "Token: ${EVALHUB_TOKEN:0:20}..."
```

Then start Jupyter with these variables available:
```bash
jupyter notebook client_usage.ipynb
```

### 2. OpenShift authentication methods

| Method | When to use | How |
|---|---|---|
| **ServiceAccount token** | Pods running inside OpenShift | Auto-detected at `/var/run/secrets/kubernetes.io/serviceaccount/token` |
| **User token** | Local development | `auth_token=os.getenv("EVALHUB_TOKEN")` (from `oc whoami -t`) |
| **Token file** | CI / automation | `auth_token_path="/path/to/token"` |

In [1]:
import os

from evalhub import (
    AsyncEvalHubClient,
    SyncEvalHubClient,
    JobSubmissionRequest,
    BenchmarkConfig,
    ModelConfig,
)

# ── Configuration ───────────────────────────────────────────────────────
EVALHUB_NAMESPACE = "opendatahub"  # Namespace where EvalHub is deployed
TENANT = "test"                    # Tenant namespace for multi-tenancy

# ── Resolve authentication token ───────────────────────────────────────
token = os.getenv("EVALHUB_TOKEN")
if not token:
    raise ValueError(
        "EVALHUB_TOKEN is not set. Run: export EVALHUB_TOKEN=$(oc whoami -t)"
    )

# ── Discover the EvalHub route ─────────────────────────────────────────
evalhub_url = os.getenv("EVALHUB_URL")
if not evalhub_url:
    host = !oc get route evalhub -n {EVALHUB_NAMESPACE} -o jsonpath='{{.spec.host}}'
    evalhub_url = f"https://{host[0]}"

print(f"EvalHub URL : {evalhub_url}")
print(f"Tenant      : {TENANT}")
print(f"Token       : {token[:20]}...")

EvalHub URL : https://evalhub-opendatahub.apps.rosa.trustyai-rui-2.2tbq.p3.openshiftapps.com
Tenant      : test
Token       : sha256~_cy5dsPnI5YdU...


## Example 1: Health Check

In [2]:
with SyncEvalHubClient(
    base_url=evalhub_url,
    auth_token=token,
    tenant=TENANT,
    insecure=True,
) as client:
    health = client.health()
    print(f"✓ EvalHub is healthy: {health['status']}")

TLS verification disabled - skipping CA bundle detection
TLS verification disabled (insecure mode)


✓ EvalHub is healthy: healthy


## Example 2: List Providers and Benchmarks

All requests automatically include `X-Tenant: test` from the client default.

In [3]:
with SyncEvalHubClient(
    base_url=evalhub_url,
    auth_token=token,
    tenant=TENANT,
    insecure=True,
) as client:
    # List all providers
    providers = client.providers.list()
    print(f"✓ Found {len(providers)} providers:")
    for provider in providers:
        print(f"  - {provider.resource.id}: {provider.name}")

    # List benchmarks from a specific provider
    if providers:
        provider_id = providers[0].resource.id
        benchmarks = client.benchmarks.list(provider_id=provider_id)
        print(f"\n✓ Found {len(benchmarks)} benchmarks for {provider_id}:")
        for benchmark in benchmarks[:5]:
            print(f"  - {benchmark.id}: {benchmark.name}")

TLS verification disabled - skipping CA bundle detection
TLS verification disabled (insecure mode)


✓ Found 6 providers:
  - garak-kfp: Garak (KFP)
  - garak: Garak
  - guidellm: GuideLLM
  - lighteval: Lighteval
  - lm_evaluation_harness: LM Evaluation Harness
  - mteb: MTEB

✓ Found 12 benchmarks for garak-kfp:
  - quick: Quick Scan
  - owasp_llm_top10: OWASP LLM Top 10
  - avid: AVID Taxonomy
  - avid_security: AVID Security
  - avid_ethics: AVID Ethics


## Example 3: Submit an Evaluation Job

The `model.url` should be the **base URL only** (host:port). The `/v1/completions` endpoint path is added automatically by the evaluation framework.

```bash
# Find available model services in your tenant namespace
oc get svc -n test
```

In [4]:
with SyncEvalHubClient(
    base_url=evalhub_url,
    auth_token=token,
    tenant=TENANT,
    insecure=True,
) as client:
    model = ModelConfig(
        url="http://vllm-server.test.svc.cluster.local:8000",
        name="tinyllama",
    )

    benchmark = BenchmarkConfig(
        id="arc_easy",
        provider_id="lm_evaluation_harness",
        parameters={
            "limit": 5,
            "tokenizer": "google/flan-t5-small",
        },
    )

    job_request = JobSubmissionRequest(model=model, benchmarks=[benchmark])
    job = client.jobs.submit(job_request)

    submitted_job_id = job.id

    print(f"✓ Job submitted: {submitted_job_id}")
    print(f"  State: {job.state}")
    print(f"  Created: {job.resource.created_at}")

TLS verification disabled - skipping CA bundle detection
TLS verification disabled (insecure mode)


✓ Job submitted: bf354c53-e1f5-42cc-8a2d-808cdad1e5b2
  State: JobStatus.PENDING
  Created: 2026-03-08 00:59:45.098162+00:00


## Example 4: Async Client

Same method names — just `await` them.

In [5]:
async with AsyncEvalHubClient(
    base_url=evalhub_url,
    auth_token=token,
    tenant=TENANT,
    insecure=True,
) as client:
    health = await client.health()
    print(f"✓ Async health check: {health['status']}")

    providers = await client.providers.list()
    print(f"✓ Found {len(providers)} providers (async)")

    if providers:
        provider_id = providers[0].resource.id
        benchmarks = await client.benchmarks.list(provider_id=provider_id)
        print(f"✓ Found {len(benchmarks)} benchmarks for {provider_id} (async)")

TLS verification disabled - skipping CA bundle detection
TLS verification disabled (insecure mode)


✓ Async health check: healthy
✓ Found 6 providers (async)
✓ Found 12 benchmarks for garak-kfp (async)


## Example 5: Wait for Completion and Retrieve Results

Poll a submitted job until it reaches a terminal state.

In [6]:
with SyncEvalHubClient(
    base_url=evalhub_url,
    auth_token=token,
    tenant=TENANT,
    insecure=True,
) as client:
    print(f"Waiting for job {submitted_job_id} to complete...")

    completed_job = client.jobs.wait_for_completion(
        job_id=submitted_job_id,
        timeout=600,
        poll_interval=5.0,
    )

    print(f"✓ Job completed with state: {completed_job.state}")

    if completed_job.results and completed_job.results.benchmarks:
        for bench in completed_job.results.benchmarks:
            print(f"\n── {bench.id} (provider: {bench.provider_id}) ──")

            if bench.metrics:
                for name, value in bench.metrics.items():
                    print(f"  {name}: {value}")

            if bench.artifacts:
                print(f"  artifacts: {list(bench.artifacts.keys())}")

            if bench.mlflow_run_id:
                print(f"  mlflow run: {bench.mlflow_run_id}")

            if bench.logs_path:
                print(f"  logs: {bench.logs_path}")

        if completed_job.results.mlflow_experiment_url:
            print(f"\nMLFlow experiment: {completed_job.results.mlflow_experiment_url}")
    else:
        print("  No results available yet")

TLS verification disabled - skipping CA bundle detection
TLS verification disabled (insecure mode)


Waiting for job bf354c53-e1f5-42cc-8a2d-808cdad1e5b2 to complete...
✓ Job completed with state: JobStatus.COMPLETED

── arc_easy (provider: lm_evaluation_harness) ──
  acc: 0.255050505050505
  acc_norm: 0.25757575757575757
  acc_norm_stderr: 0.008973187820215224
  acc_stderr: 0.008944265906130712


## Example 6: Per-Request Tenant Override

The default tenant is set at client creation, but any resource method accepts a
`tenant=` keyword to override it for a single call.

In [7]:
with SyncEvalHubClient(
    base_url=evalhub_url,
    auth_token=token,
    tenant=TENANT,  # default: "test"
    insecure=True,
) as client:
    # Uses the default tenant ("test")
    jobs_test = client.jobs.list()
    print(f"✓ Jobs in '{TENANT}': {len(jobs_test)}")

    # Override for a single call
    OTHER_TENANT = "team-b"
    try:
        jobs_other = client.jobs.list(tenant=OTHER_TENANT)
        print(f"✓ Jobs in '{OTHER_TENANT}': {len(jobs_other)}")
    except Exception as e:
        # Expected if the token doesn't have access to 'team-b'
        print(f"✗ Access to '{OTHER_TENANT}' denied (expected): {e}")

TLS verification disabled - skipping CA bundle detection
TLS verification disabled (insecure mode)


✓ Jobs in 'test': 1
✓ Jobs in 'team-b': 0
